# EXP-009: PLR (Periodic Linear Representation) - MLP only

**근거:** RealMLP의 핵심 차별 기법인 **PLR encoding**의 단독 효과만 빠르게 측정. XGB/LGB/CAT 학습 생략 (EXP-008 결과 메모리에).

**PLR 원리:** 수치형 feature x를 학습 가능한 frequencies `c`로 sin/cos 주기 인코딩:
```
phi_periodic(x) = [sin(2π·c_i·x), cos(2π·c_i·x)]  for k frequencies
phi_PLR(x) = GELU(Linear(phi_periodic(x)))
```
→ NN이 미세 비선형 패턴 잡기 쉬워짐 (Gorishniy et al. 2022)

**변경점 vs EXP-008 MLP:** Numeric input pipeline에 PLR module 추가. 나머지 동일 (epochs 80, lr 1e-3, width 256-128-64, dropout 0.2, single seed).

**비교 지표**: MLP solo OOF (EXP-008 = 0.94095)와 직접 비교.

In [1]:
import pandas as pd
import numpy as np
import time
import math
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
import warnings
warnings.filterwarnings('ignore')

train = pd.read_csv('../data/train.csv')
test  = pd.read_csv('../data/test.csv')

TARGET = 'PitNextLap'
CAT_COLS = ['Driver', 'Compound', 'Race']
NUM_COLS = ['Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position',
            'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation',
            'RaceProgress', 'Position_Change']
y = train[TARGET].astype(int).values

train_le = train.copy(); test_le = test.copy()
for c in CAT_COLS:
    le = LabelEncoder()
    train_le[c] = le.fit_transform(train_le[c].astype(str))
    seen = set(le.classes_)
    test_le[c] = test_le[c].astype(str).map(lambda v: le.transform([v])[0] if v in seen else -1)

drop_cols = ['id', TARGET]
feature_cols = [c for c in train.columns if c not in drop_cols]
X_le, X_test_le = train_le[feature_cols], test_le[feature_cols]

N_FOLDS, SEED = 5, 42
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
splits = list(skf.split(X_le, y))
print(f'Train: {train.shape}, Test: {test.shape}')

Train: (439140, 16), Test: (188165, 15)


## 1. PLR Encoder 직접 구현

- `n_frequencies`(k): 각 numeric feature당 학습 가능한 frequency 개수 (RealMLP: 16)
- `hidden_dim`: linear 출력 차원 (RealMLP: 8)
- `sigma`: frequency 초기화 std (RealMLP: 2.33)

각 numeric feature → 2k periodic features (sin+cos) → linear → hidden_dim
결과 shape: (B, n_numeric × hidden_dim)

In [2]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

torch.backends.cudnn.benchmark = True
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

class PLREncoder(nn.Module):
    """Periodic Linear Representation for numeric features.

    Args:
        n_numeric: number of numeric features
        k: number of periodic frequencies per feature
        hidden: per-feature output dim after linear+activation
        sigma: initial std for frequencies
    """
    def __init__(self, n_numeric, k=16, hidden=8, sigma=2.33):
        super().__init__()
        # Learnable frequencies, shape: (n_numeric, k)
        self.freqs = nn.Parameter(torch.randn(n_numeric, k) * sigma)
        # Shared linear: 2k (sin/cos) -> hidden
        self.linear = nn.Linear(2 * k, hidden)
        self.act = nn.GELU()
        self.n_numeric = n_numeric
        self.hidden = hidden

    def forward(self, x_num):
        # x_num: (B, n_numeric)
        # → (B, n_numeric, 1) * (1, n_numeric, k) = (B, n_numeric, k)
        z = 2 * math.pi * x_num.unsqueeze(-1) * self.freqs.unsqueeze(0)
        sin_z, cos_z = torch.sin(z), torch.cos(z)
        # Concat along last dim → (B, n_numeric, 2k)
        p = torch.cat([sin_z, cos_z], dim=-1)
        # Linear over 2k → hidden, then activation
        out = self.act(self.linear(p))  # (B, n_numeric, hidden)
        return out.flatten(start_dim=1)  # (B, n_numeric * hidden)


# Quick sanity test
plr_test = PLREncoder(11, k=16, hidden=8).to(device)
x_test = torch.randn(4, 11, device=device)
out = plr_test(x_test)
print(f'PLR output shape: {out.shape} (expected (4, 88))')
del plr_test, x_test, out

device: cuda
PLR output shape: torch.Size([4, 88]) (expected (4, 88))


## 2. PLR-MLP 모델 정의

- Cat embedding 28 + Raw numeric 11 (BN) + **PLR features 88** = 127-dim input
- Body: `[127] → 256 → 128 → 64 → 1` (GELU + BN + Dropout 0.2)

In [3]:
class PLRMLP(nn.Module):
    def __init__(self, cat_cards, emb_dims, n_numeric, plr_k=16, plr_hidden=8,
                 hidden=(256, 128, 64), dropout=0.2):
        super().__init__()
        # Cat embeddings
        self.embs = nn.ModuleList([nn.Embedding(card, dim) for card, dim in zip(cat_cards, emb_dims)])
        emb_total = sum(emb_dims)
        # Raw numeric BatchNorm
        self.bn_num = nn.BatchNorm1d(n_numeric)
        # PLR encoder
        self.plr = PLREncoder(n_numeric, k=plr_k, hidden=plr_hidden, sigma=2.33)
        plr_total = n_numeric * plr_hidden
        # Body
        in_dim = emb_total + n_numeric + plr_total
        layers = []
        prev = in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.GELU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.body = nn.Sequential(*layers)
        self.in_dim = in_dim
        print(f'PLRMLP input dim: emb={emb_total} + raw_num={n_numeric} + plr={plr_total} = {in_dim}')

    def forward(self, x_cat, x_num):
        emb = torch.cat([emb_layer(x_cat[:, i]) for i, emb_layer in enumerate(self.embs)], dim=1)
        raw_num = self.bn_num(x_num)
        plr_feat = self.plr(x_num)  # PLR uses raw numeric (already scaled outside)
        x = torch.cat([emb, raw_num, plr_feat], dim=1)
        return self.body(x).squeeze(1)

## 3. 5-fold 학습

In [4]:
# Cat: -1 → 0 (unseen), 0..N-1 → 1..N
cat_cards = []
X_cat_tr = np.zeros((len(X_le), len(CAT_COLS)), dtype=np.int64)
X_cat_te = np.zeros((len(X_test_le), len(CAT_COLS)), dtype=np.int64)
for i, c in enumerate(CAT_COLS):
    n_unique = X_le[c].max() + 1
    cat_cards.append(n_unique + 1)
    X_cat_tr[:, i] = X_le[c].values + 1
    te_vals = X_test_le[c].values + 1
    te_vals[te_vals < 0] = 0
    X_cat_te[:, i] = te_vals

X_num_tr_raw = X_le[NUM_COLS].values.astype(np.float32)
X_num_te_raw = X_test_le[NUM_COLS].values.astype(np.float32)

EMB_DIMS = [16, 4, 8]
BATCH_SIZE = 4096
MAX_EPOCHS = 80
PATIENCE = 20

oof_mlp = np.zeros(len(X_le))
test_mlp = np.zeros(len(X_test_le))
mlp_epochs_used = []
mlp_best_aucs = []
t0 = time.time()

for fold, (tr_idx, va_idx) in enumerate(splits):
    scaler = StandardScaler()
    X_num_tr = scaler.fit_transform(X_num_tr_raw[tr_idx])
    X_num_va = scaler.transform(X_num_tr_raw[va_idx])
    X_num_te = scaler.transform(X_num_te_raw)

    Xc_tr = torch.tensor(X_cat_tr[tr_idx], dtype=torch.long)
    Xn_tr = torch.tensor(X_num_tr, dtype=torch.float32)
    yt_tr = torch.tensor(y[tr_idx], dtype=torch.float32)
    Xc_va = torch.tensor(X_cat_tr[va_idx], dtype=torch.long, device=device)
    Xn_va = torch.tensor(X_num_va, dtype=torch.float32, device=device)
    yt_va = y[va_idx]
    Xc_te = torch.tensor(X_cat_te, dtype=torch.long, device=device)
    Xn_te = torch.tensor(X_num_te, dtype=torch.float32, device=device)

    train_loader = DataLoader(
        TensorDataset(Xc_tr, Xn_tr, yt_tr),
        batch_size=BATCH_SIZE, shuffle=True, drop_last=False,
        num_workers=0, pin_memory=True,
    )

    torch.manual_seed(SEED)
    model = PLRMLP(cat_cards, EMB_DIMS, len(NUM_COLS)).to(device)
    optim = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=MAX_EPOCHS)
    bce = nn.BCEWithLogitsLoss()

    best_auc = -1
    best_state = None
    best_epoch = 0
    wait = 0
    epochs_used = 0

    for epoch in range(MAX_EPOCHS):
        model.train()
        for xc, xn, yb in train_loader:
            xc = xc.to(device, non_blocking=True)
            xn = xn.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)
            optim.zero_grad()
            logit = model(xc, xn)
            loss = bce(logit, yb)
            loss.backward()
            optim.step()
        sched.step()

        model.eval()
        with torch.no_grad():
            va_logit = model(Xc_va, Xn_va).cpu().numpy()
        va_prob = 1 / (1 + np.exp(-va_logit))
        auc = roc_auc_score(yt_va, va_prob)
        epochs_used = epoch + 1

        if auc > best_auc + 1e-6:
            best_auc = auc
            best_epoch = epochs_used
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= PATIENCE:
                break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        va_logit = model(Xc_va, Xn_va).cpu().numpy()
        te_logit = model(Xc_te, Xn_te).cpu().numpy()
    oof_mlp[va_idx] = 1 / (1 + np.exp(-va_logit))
    test_mlp += (1 / (1 + np.exp(-te_logit))) / N_FOLDS
    mlp_epochs_used.append(epochs_used)
    mlp_best_aucs.append(best_auc)
    early_stopped = epochs_used < MAX_EPOCHS
    print(f'PLR-MLP fold {fold}: best AUC={best_auc:.5f}, best_epoch={best_epoch}, epochs_used={epochs_used} {"(early stop)" if early_stopped else "(max)"}')

    del model, optim, sched, Xc_va, Xn_va, Xc_te, Xn_te
    torch.cuda.empty_cache()

plr_mlp_oof_auc = roc_auc_score(y, oof_mlp)
print(f'\nPLR-MLP OOF AUC: {plr_mlp_oof_auc:.5f}')
print(f'  vs EXP-008 MLP 0.94095: {plr_mlp_oof_auc-0.94095:+.5f}')
print(f'  vs EXP-007 MLP 0.94044: {plr_mlp_oof_auc-0.94044:+.5f}')
print(f'  elapsed: {time.time()-t0:.1f}s')

PLRMLP input dim: emb=28 + raw_num=11 + plr=88 = 127
PLR-MLP fold 0: best AUC=0.94611, best_epoch=17, epochs_used=37 (early stop)
PLRMLP input dim: emb=28 + raw_num=11 + plr=88 = 127
PLR-MLP fold 1: best AUC=0.94367, best_epoch=14, epochs_used=34 (early stop)
PLRMLP input dim: emb=28 + raw_num=11 + plr=88 = 127
PLR-MLP fold 2: best AUC=0.94506, best_epoch=19, epochs_used=39 (early stop)
PLRMLP input dim: emb=28 + raw_num=11 + plr=88 = 127
PLR-MLP fold 3: best AUC=0.94436, best_epoch=19, epochs_used=39 (early stop)
PLRMLP input dim: emb=28 + raw_num=11 + plr=88 = 127
PLR-MLP fold 4: best AUC=0.94531, best_epoch=14, epochs_used=34 (early stop)

PLR-MLP OOF AUC: 0.94486
  vs EXP-008 MLP 0.94095: +0.00391
  vs EXP-007 MLP 0.94044: +0.00442
  elapsed: 649.5s


## 4. OOF/Test predictions 저장 + EXP-008 결과와 비교

이 노트북은 MLP 단독 학습이라 4-way blend 안 함. 대신:
- PLR-MLP OOF/test 저장 — EXP-010에서 활용 가능
- MLP solo submission 저장

In [5]:
# OOF/test predictions 저장 (EXP-010에서 활용)
np.save('../submissions/exp009_plr_mlp_oof.npy', oof_mlp)
np.save('../submissions/exp009_plr_mlp_test.npy', test_mlp)

# Submission CSV
sub = pd.DataFrame({'id': test['id'], 'PitNextLap': test_mlp})
sub.to_csv('../submissions/submission_exp009_plr_mlp.csv', index=False)
print('saved exp009_plr_mlp_oof.npy / exp009_plr_mlp_test.npy / submission_exp009_plr_mlp.csv')
print(f'test pred mean: {test_mlp.mean():.4f}')
print()
print('=== EXP-009 결론 ===')
print(f'  EXP-007 MLP solo: 0.94044')
print(f'  EXP-008 MLP solo: 0.94095  (epochs 50→80)')
print(f'  EXP-009 PLR-MLP : {plr_mlp_oof_auc:.5f}  (+PLR encoding)')
print()
delta = plr_mlp_oof_auc - 0.94095
if delta > 0.003:
    print(f'  → PLR 효과 큼 (+{delta:.5f}). EXP-010에서 4-way 재학습 + 다른 기법 추가 가치 있음')
elif delta > 0.001:
    print(f'  → PLR 효과 중간 (+{delta:.5f}). 다른 기법과 조합으로 시너지 가능성 점검')
elif delta > -0.001:
    print(f'  → PLR 효과 미미 ({delta:+.5f}). RealMLP의 다른 기법 (PLR보다 다른 요소들)이 핵심일 가능성')
else:
    print(f'  → PLR 효과 negative ({delta:+.5f}). 우리 baseline에 PLR 부적합. 우선순위 재고')

saved exp009_plr_mlp_oof.npy / exp009_plr_mlp_test.npy / submission_exp009_plr_mlp.csv
test pred mean: 0.2010

=== EXP-009 결론 ===
  EXP-007 MLP solo: 0.94044
  EXP-008 MLP solo: 0.94095  (epochs 50→80)
  EXP-009 PLR-MLP : 0.94486  (+PLR encoding)

  → PLR 효과 큼 (+0.00391). EXP-010에서 4-way 재학습 + 다른 기법 추가 가치 있음
